# 03 · Paper-level splitting and leakage audit

**Programming for AI · Topic 26 · Participant 1 · Nulimu Arey · AAI-2606M**

Each paper and both language versions stay in one partition. Retain only scored reviews for training, validation and test.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "prepare_data.py").is_file():
    raise FileNotFoundError("Open this notebook from the project root or the notebooks directory.")
sys.path.insert(0, str(root))

In [2]:
from IPython.display import display
from src.data_collection import load_sources, pair_reviews
from src.data_leakage import audit_overlap, duplicate_groups, find_duplicate_edges, group_split, random_row_split, verify_splits
from prepare_data import prepare, split_summary

reviews = pair_reviews(load_sources())

## 1. Build paper groups and check duplicate content

Start with paper identities based on normalized titles. Compare exact titles and abstracts within each language. Compare all distinct cross-paper abstracts with character 5-gram Jaccard similarity at a threshold of 0.80. If an edge is found, keep all papers in the connected component together. This rule detects highly similar text, while paraphrased versions may score lower.

In [3]:
edges = find_duplicate_edges(reviews)
groups = duplicate_groups(reviews["paper_id"], edges)
print(f"Distinct papers: {reviews['paper_id'].nunique()}")
print(f"Cross-paper exact or near match rows: {len(edges)}")
display(edges)

Distinct papers: 215
Cross-paper exact or near match rows: 0


,paper_a,paper_b,language,check,similarity


## 2. Assign groups and compare with a row shuffle

Select a reproducible 70/15/15 target split by choosing the most balanced of 2,000 candidate group assignments using seed 26. All three labels must appear in each split. The random-row example below uses the same review counts in each partition.

In [4]:
data = group_split(reviews, groups)
grouped = audit_overlap(data, data["split"], edges)
row_shuffle = audit_overlap(data, random_row_split(data), edges)
verify_splits(reviews, data, grouped)

display(split_summary(data))
display(row_shuffle[["split_a", "split_b", "paper_id"]].rename(columns={"paper_id": "Shared paper IDs after row shuffle"}))
display(grouped)

label,Review pairs,Papers,Low,Moderate,High
Partition,,,,,
train,270,130,31,168,71
validation,57,27,6,35,16
test,58,28,7,35,16


,split_a,split_b,Shared paper IDs after row shuffle
0,train,validation,40
1,train,test,37
2,validation,test,11


,split_a,split_b,paper_id,split_group_id,title_en,abstract_en,normalized_title_en,title_kk,abstract_kk,normalized_title_kk,near_abstract_en,near_abstract_kk
0,train,validation,0,0,0,0,0,0,0,0,0,0
1,train,test,0,0,0,0,0,0,0,0,0,0
2,validation,test,0,0,0,0,0,0,0,0,0,0


## 3. Export the audited handoff

Run the shared pipeline to write the processed corpus, the three split CSVs, the data quality report, the EDA figure and the full overlap audit. Both languages use these same partitions.

In [5]:
prepare()

label       Review pairs  Papers  Low  Moderate  High
Partition                                            
train                270     130   31       168    71
validation            57      27    6        35    16
test                  58      28    7        35    16
Wrote the dataset, split files, audit, and corpus_overview.png
